In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from collections import defaultdict
import pyfastx
import numpy as np
from sklearn.model_selection import train_test_split
from numpy import array
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import resample
#from window_slider import Slider
from Bio.Seq import Seq
import os
from pathlib import Path

In [2]:
gisaid_fa_1 = pyfastx.Fasta('/home/hnguyen/Documents/PhD/IAV_Data/GISAID/data/gisaid.fasta', build_index=True)
# Create seq id dict
gisaid_seq_id_dict_1 = defaultdict()
for seq in gisaid_fa_1:
    seq_id = seq.name.split('|')[0]
    gisaid_seq_id_dict_1[seq_id] = seq.name

In [3]:
gisaid_fa_2 = pyfastx.Fasta('/home/hnguyen/Documents/PhD/IAV_Data/GISAID_2024_2025/sequences/RNA.fasta', build_index=True)
# Create seq id dict
gisaid_seq_id_dict_2 = defaultdict()
for seq in gisaid_fa_2:
    seq_id = seq.name.split('|')[0]
    gisaid_seq_id_dict_2[seq_id] = seq.name

In [4]:
def get_file_stats(filepath, segment, subtype_list=None):
    """Reads a file and returns a dictionary of stats."""
    if subtype_list is None:
        subtype_list = ['H1N1', 'H3N2', 'H1N2', 'H2N2', 'H3N8', 'H7N9', 'H9N2', 'H5N6', 'H5N1']

    stats = {
        "count": 0,
        "year_min": "N/A",
        "year_max": "N/A",
        "amb_min": "N/A",
        "amb_max": "N/A",
        "hosts": {},
        "len_min": "N/A",
        "len_max": "N/A",
        "subtype_counts": {subtype: 0 for subtype in subtype_list},
        "subtype_percs": {subtype: 0.0 for subtype in subtype_list},
        "clade_counts": {"6B.1A.5a.2a": 0, "6B.1A.5a.2a.1": 0},
        "clade_percs": {"6B.1A.5a.2a": 0.0, "6B.1A.5a.2a.1": 0.0}
    }

    seq_list = set()

    try:
        df = pd.read_csv(filepath)
        df = df.reset_index(drop=True)

        for index, row in df.iterrows():
            seq_id = row[segment[1:] + '_ID']
            if seq_id in gisaid_seq_id_dict_1:
                seq_list.add(str(gisaid_fa_1[gisaid_seq_id_dict_1[seq_id]].seq).upper())
            else:
                seq_list.add(str(gisaid_fa_2[gisaid_seq_id_dict_2[seq_id]].seq).upper())

        print(f"{filepath}:", df.shape, len(seq_list))

        if 'Collection_Year' not in df.columns or 'Host_ID' not in df.columns:
            return stats, seq_list

        df['Collection_Year'] = pd.to_numeric(df['Collection_Year'], errors='coerce')
        df = df.dropna(subset=['Collection_Year'])

        if df.empty:
            return stats, seq_list

        total_seqs = len(df)
        stats["count"] = total_seqs
        stats["year_min"] = int(df['Collection_Year'].min())
        stats["year_max"] = int(df['Collection_Year'].max())

        if 'Ambiguous_Count' in df.columns:
            stats["amb_min"] = int(df['Ambiguous_Count'].min())
            stats["amb_max"] = int(df['Ambiguous_Count'].max())

        if 'Seq_Len' in df.columns:
            stats["len_min"] = int(df['Seq_Len'].min())
            stats["len_max"] = int(df['Seq_Len'].max())

        stats["hosts"] = df['Host_ID'].value_counts().sort_index().to_dict()

        # Subtype breakdown
        if 'Subtype' in df.columns:
            subtype_df = df[df['Subtype'].isin(subtype_list)]
            subtype_counts = subtype_df['Subtype'].value_counts().to_dict()

            for subtype in subtype_list:
                count = subtype_counts.get(subtype, 0)
                stats["subtype_counts"][subtype] = count
                if total_seqs > 0:
                    stats["subtype_percs"][subtype] = (count / total_seqs) * 100

        # Clade counts for H1N1 only
        if 'Clade' in df.columns and 'Subtype' in df.columns:
            df_h1n1_seasonal = df[
                (df['Subtype'] == 'H1N1') &
                df['Clade'].isin(["6B.1A.5a.2a", "6B.1A.5a.2a.1"])
            ]
            clade_counts = df_h1n1_seasonal['Clade'].value_counts()

            c1 = clade_counts.get("6B.1A.5a.2a", 0)
            c2 = clade_counts.get("6B.1A.5a.2a.1", 0)
            stats["clade_counts"]["6B.1A.5a.2a"] = c1
            stats["clade_counts"]["6B.1A.5a.2a.1"] = c2

            if total_seqs > 0:
                stats["clade_percs"]["6B.1A.5a.2a"] = (c1 / total_seqs) * 100
                stats["clade_percs"]["6B.1A.5a.2a.1"] = (c2 / total_seqs) * 100

    except FileNotFoundError:
        pass

    return stats, seq_list

In [6]:
subtype_list = ['H1N1', 'H3N2', 'H1N2', 'H2N2', 'H3N8', 'H7N7', 'H7N9', 'H9N2', 'H5N6', 'H5N1']
segments = ["01_PB2", "02_PB1", "03_PA", "04_HA", "05_NP", "06_NA", "07_MP", "08_NS"]

rows = []

for segment in segments:
    train_file = f"/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/{segment}/{segment}_Train_Filtered.csv"
    test_file  = f"/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/{segment}/Part1_2_{segment}_Test_Filtered.csv"

    tr, seq_tra = get_file_stats(train_file, segment, subtype_list)
    te, seq_te = get_file_stats(test_file, segment, subtype_list)

    row = {
        "Segment": segment,

        "Train_Count": tr["count"],
        "Train_Year": f"{tr['year_min']} - {tr['year_max']}",
        "Train_Ambiguous": f"{tr['amb_min']} - {tr['amb_max']}",
        "Train_Length": f"{tr['len_min']} - {tr['len_max']}",
        "Train_Hosts": str(tr["hosts"]),
        "Train_2a": tr["clade_counts"].get("6B.1A.5a.2a", 0),
        "Train_2a.1": tr["clade_counts"].get("6B.1A.5a.2a.1", 0),

        "Test_Count": te["count"],
        "Test_Year": f"{te['year_min']} - {te['year_max']}",
        "Test_Ambiguous": f"{te['amb_min']} - {te['amb_max']}",
        "Test_Length": f"{te['len_min']} - {te['len_max']}",
        "Test_Hosts": str(te["hosts"]),
        "Test_2a": te["clade_counts"].get("6B.1A.5a.2a", 0),
        "Test_2a.1": te["clade_counts"].get("6B.1A.5a.2a.1", 0),

        "Duplicated_Sequences": len(seq_tra & seq_te),
    }

    for st in subtype_list:
        row[f"Train_{st}"] = tr["subtype_counts"].get(st, 0)
        row[f"Test_{st}"] = te["subtype_counts"].get(st, 0)

    rows.append(row)

df_out = pd.DataFrame(rows)

ordered_cols = [
    "Segment",
    "Train_Count", "Train_Year", "Train_Ambiguous", "Train_Length", "Train_Hosts",
    "Train_H1N1", "Train_H3N2", "Train_H1N2", "Train_H2N2", "Train_H3N8", "Train_H7N7", "Train_H7N9", "Train_H9N2", "Train_H5N6", "Train_H5N1",
    "Train_2a", "Train_2a.1",
    "Test_Count", "Test_Year", "Test_Ambiguous", "Test_Length", "Test_Hosts",
    "Test_H1N1", "Test_H3N2", "Test_H1N2", "Test_H2N2", "Test_H3N8", "Test_H7N7", "Test_H7N9", "Test_H9N2", "Test_H5N6", "Test_H5N1",
    "Test_2a", "Test_2a.1",
    "Duplicated_Sequences"
]

df_out = df_out[ordered_cols]

sum_row = {"Segment": "ALL"}

for col in df_out.columns:
    if col.startswith("Train_") and pd.api.types.is_numeric_dtype(df_out[col]):
        sum_row[col] = df_out[col].sum()
    elif col.startswith("Test_") and pd.api.types.is_numeric_dtype(df_out[col]):
        sum_row[col] = df_out[col].sum()
    elif col == "Duplicated_Sequences":
        sum_row[col] = df_out[col].sum()
    elif col != "Segment":
        sum_row[col] = ""

summary_df = pd.DataFrame([sum_row])[ordered_cols]
final_df = pd.concat([df_out, summary_df], ignore_index=True)

out_path = Path("../results/data_distribution.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")

/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/01_PB2/01_PB2_Train_Filtered.csv: (78732, 35) 78732
/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/01_PB2/Part1_2_01_PB2_Test_Filtered.csv: (83325, 35) 83325
/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/02_PB1/02_PB1_Train_Filtered.csv: (77369, 35) 77369
/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/02_PB1/Part1_2_02_PB1_Test_Filtered.csv: (77041, 35) 77041
/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/03_PA/03_PA_Train_Filtered.csv: (77642, 35) 77642
/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/03_PA/Part1_2_03_PA_Test_Filtered.csv: (85772, 35) 85772
/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/04_HA/04_HA_Train_Filtered.csv: (148068, 35) 148068
/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/04_HA/Part1_2_04_HA_Test_Filtered.csv: (105804, 35) 105804
/home/hnguyen/Documents/PhD/Part2/Paper_02_Prepa